# FuGuard reproduction - ALL-IN-ONE Colab notebook
Everything from the numbered notebooks is in here as blocks - run them **top to bottom, block by block** (Colab's sidebar outline lists the parts).
The same blocks also exist as separate notebooks under `notebooks/`.

1. **Runtime → Change runtime type → GPU.**
2. Run *Part 0* (unzip the project to `/content/FuGuard_Reproduction`, install, smoke test).
3. Go through the parts in order. Inside an experiment part, run its blocks in order too (the config block defines the variables the run block uses).

| Part | What | Produces |
|---|---|---|
| 0 | setup, smoke test, pilot run | working environment |
| 01a-c | data (CIFAR-10 from Hugging Face `Sakhiur/blue-bird-77`) | splits, Fig. 3/4/5 |
| 02a / 02b | Table II protocol (5 recovery rounds) / Table III protocol (50) | Table II, Fig. 6 / Table III |
| 02c-02f | ablation / sensitivity / 100 clients / AWE | Table IV / Fig. 10 / Fig. 9 / Fig. 7-8 |
| 03a-b | tables, figures, download | `results/` |

Everything is stored on the Colab VM under `/content/FuGuard_Reproduction/{data,artifacts,results}`. It disappears when the runtime is recycled:
run `colab.zip_results()` (last block) regularly. Finished runs are saved immediately, so after a disconnect re-run the block and it resumes.

In [ ]:
import glob, os, zipfile
PROJECT = "/content/FuGuard_Reproduction"
if not os.path.exists(PROJECT):
    zips = sorted(glob.glob("/content/FuGuard_Reproduction*.zip"))
    if not zips:
        from google.colab import files
        files.upload()                      # pick FuGuard_Reproduction_Colab.zip
        zips = sorted(glob.glob("/content/FuGuard_Reproduction*.zip"))
    zipfile.ZipFile(zips[-1]).extractall("/content")
os.chdir(PROJECT)
print(os.getcwd(), os.listdir("."))

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
# Colab's preinstalled torchao breaks diffusers ("cannot import name 'FqnToConfig'"); we do not use it
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU, then re-run"

In [ ]:
import os, sys, pathlib, importlib.util, subprocess
_cwd = pathlib.Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents, pathlib.Path("/content/FuGuard_Reproduction")]
             if (p / "fuguard_repro").exists()), None)
if ROOT is None:
    raise RuntimeError("Project not found. On Colab: run COLAB_START_HERE.ipynb first (it unzips to /content/FuGuard_Reproduction).")
os.environ.setdefault("FUGUARD_ROOT", str(ROOT))
sys.path.insert(0, str(ROOT))
for _mod in ("geomloss", "diffusers"):          # a Colab runtime restart wipes pip installs -> re-install if missing
    if importlib.util.find_spec(_mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
        break
print("project root:", ROOT)

# Part 0 · Environment check and smoke test

**Colab:** use `COLAB_START_HERE.ipynb` for the first-time setup (unzip, install, GPU check). This notebook can be re-run any time.

In [ ]:
import torch
from fuguard_repro.paths import ensure_dirs, ROOT, DATA_DIR, RAW_DIR, TABLES_DIR, FIGURES_DIR
ensure_dirs()
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(CPU only - real runs will be very slow)"))
from fuguard_repro import colab
colab.status()

In [ ]:
from fuguard_repro.smoke import run_smoke_test
df = run_smoke_test(verbose=False)
df

### Can this machine download the VAE? (needed once for CIFAR-10/100, SVHN, EuroSAT FuGuard runs)

In [ ]:
from fuguard_repro.config import BASE_IMAGE_CONFIG
from fuguard_repro.proxy import load_vae
vae = load_vae(BASE_IMAGE_CONFIG["vae_id"], "cpu")      # needs internet / HF access the first time
print("VAE loaded:", type(vae).__name__)
del vae

# Pilot run (recommended before the big grids): CIFAR-10, seed 0, FedSGA + FuGuard
Downloads CIFAR-10 from Hugging Face, builds the split, and runs the Table II protocol for two methods.

In [ ]:
from fuguard_repro.config import image_config
from fuguard_repro.data_image import prepare_image_data
prepare_image_data(image_config("CIFAR10"), 0)        # fetches Sakhiur/blue-bird-77 automatically

In [ ]:
from fuguard_repro.runner import run_grid, standard_jobs
run_grid("table2", ["CIFAR10"], standard_jobs(["fedsga", "fuguard"]), seeds=(0,), recovery_rounds=5)

### Optional: long runs in a background process
`colab.run_in_background` starts `scripts/run_experiment.py` separately (survives a closed browser tab, not a recycled runtime). Data must already be
prepared (Part 01a or `scripts/prepare_data.py`). Run one GPU job at a time.

In [ ]:
# p = colab.run_in_background(["table2", "--datasets", "CIFAR10", "--seeds", "0", "1"], log="logs/table2_cifar10.log")
# colab.tail_log("logs/table2_cifar10.log")

# 01a · Prepare the four image benchmarks (Dirichlet split + backdoor)
Port of `main_image/prepare_data.py`. For each dataset and seed: 10 clients, Dirichlet(α) label skew
(α=0.1; CIFAR-100 α=1), attacker = `client_3`, 4×4 white patch → label 9 on **all** its samples.
Test sets: clean + triggered (target-class images are left untouched, as in the official code).

* **CIFAR-10 comes from the Hugging Face dataset `Sakhiur/blue-bird-77` (folder `cifar-10/`)**: `cifar-10-batches-py` (~186 MB) is
  downloaded into `data/` and read directly (torchvision's downloader is only the fallback if Hugging Face fails).
* SVHN / CIFAR-100 download automatically. **EuroSAT** is looked up in `data/EuroSAT/2750`
  (or `data/eurosat/2750`); otherwise torchvision's downloader is tried, then you get instructions.
* Disk: roughly 10 GB for everything below (fp32 tensors; estimate from tensor sizes, not measured).
* Data are split once per seed, so seeds 0 and 1 give different partitions (as in the paper's two runs).

In [ ]:
from fuguard_repro import hf_data
hf_data.fetch_cifar10()      # Sakhiur/blue-bird-77 -> data/cifar-10-batches-py (skipped if already there)

In [ ]:
from fuguard_repro.config import IMAGE_DATASETS, PROTOCOLS, image_config
from fuguard_repro.data_image import prepare_image_data
from fuguard_repro.paths import ensure_dirs
ensure_dirs()

DATASETS = IMAGE_DATASETS            # e.g. ["CIFAR10"] to start small
SEEDS = (0, 1)
for ds in DATASETS:
    for seed in SEEDS:
        prepare_image_data(image_config(ds), seed)

### 100-client SVHN (Fig. 9 / scalability)
The paper does not state α or the attacker index for this run; we assume the SVHN α=0.1 and `client_3`.
With 100 clients and α=0.1 some clients can be (nearly) empty - the engine skips empty clients and tells you if the attacker is empty.

In [ ]:
for seed in (0, 1):
    prepare_image_data(image_config("SVHN", num_clients=100, tag="SVHN-100"), seed)

# 01b · Prepare the AWE industrial dataset (6 clients, spike backdoor)
The dataset repo has this layout (adapted to in this project): `Dataset/normal` (7 files, state 0), `Dataset/1chip` (8 files),
`Dataset/2chips … 5chips` (8 files each). The official FuGuard code looked for folders `0chips…5chips`, so it could not read this layout
(and silently skipped missing files); `awe_prep.py` maps `normal → state 0`, `1chip → state 1`, … and matches names case-insensitively
(`Normal_0.json` vs `normal_1.json`).

Get the data on Colab (about 2 GB, stored on the VM): `colab.fetch_awe_dataset()` clones the GitHub repo, or pass your own copy:
`colab.fetch_awe_dataset("/content/drive/MyDrive/NGIT-2022-AWE-Industrial-Dataset-master.zip")` (a zip or a folder).

Pipeline (unchanged from the repo): z-score → 4th-order low-pass (250 Hz) → downsample 4 kHz→500 Hz → windows of 30 rows (stride 10) × 1000 features.
Client *k* holds mostly state *k* (groups 0-3) plus random other groups. Attacker `client_2`: spike trigger (+20 on several positions, noise σ=0.5)
on **all** its windows → class 5. Groups 0-6 train (state 0: 0-5) and the last file of each state is the test set.

Fixes vs the repo: config keys `SAVE_FOLDER / CLIENTS / SEED` did not exist (KeyError); trigger noise uses a seeded RNG; `num_clients` is 6.
Tested end-to-end on a *fake* tree that mirrors the real listing; the JSON field names `vibr_UB` / `vibr_LB` come from the authors' code and are not verified on the real files
(if they differ you get an error that prints the keys found - then open the dataset's `Read_Data.py`).

In [ ]:
from fuguard_repro import colab
colab.fetch_awe_dataset()      # or colab.fetch_awe_dataset("/path/to/zip-or-folder")

In [ ]:
from fuguard_repro.config import awe_config
from fuguard_repro.data_awe import prepare_awe_data
prepare_awe_data(awe_config())

# 01c · Data figures (Fig. 3, Fig. 4, Fig. 5)
Drawn from the prepared data. Outputs go to `results/figures/`.

In [ ]:
import numpy as np
from IPython.display import Image, display
from fuguard_repro.config import image_config, awe_config
from fuguard_repro.data_image import client_class_counts, test_class_counts
from fuguard_repro import plotting

cfg = image_config("SVHN")
before = client_class_counts(cfg, 0, before_injection=True)
after = client_class_counts(cfg, 0, before_injection=False)
p3 = plotting.plot_label_distribution(before, after, "SVHN, 10 clients")
clean, poison = test_class_counts(cfg, 0)
p4 = plotting.plot_test_distribution(clean, poison, "SVHN test sets")
display(Image(str(p3))); display(Image(str(p4)))

### Fig. 5 · AWE sample before / after trigger (needs 01b)

In [ ]:
from fuguard_repro.data_awe import awe_dir
d = np.load(awe_dir(awe_config()) / "global_test.npz")
i = 0   # first non-target test window; X_backdoor[i] is its triggered version
mask = d["y_clean"] != awe_config()["target_label"]
clean_w, trig_w = d["X_clean"][mask][i][0], d["X_backdoor"][i][0]     # first row of the 30x1000 window
display(Image(str(plotting.plot_awe_trigger(clean_w, trig_w))))

# 02a · Protocol A - 50 training rounds + **5 recovery rounds**
Produces the raw data for **Table II** (ACC/ASR at the end of 5 recovery rounds), **Fig. 6** (ACC/ASR/MIA curves),
and it is also the source of the λ_OT=0.1 point of Table IV and Fig. 10.
Paper settings: 4 datasets × 6 methods × seeds {0, 1}.  Local optimizer: Adam (kept from the repo on purpose).

Resumable: finished (dataset, method, seed) results in `results/raw/` are skipped, so you can stop and re-run.
The 49 shared training rounds are trained **once per (dataset, seed)** and cached in `artifacts/checkpoints/`;
every method then starts from the same pre-unlearning model (identical to what the repo's scripts recompute each time).

Tip: validate the pipeline first with `DATASETS=["CIFAR10"]`, `SEEDS=(0,)`.
**Colab sessions:** the VM is wiped when the runtime is recycled and free sessions are time-limited. Every finished run is saved at once, so just re-run the cell after a disconnect;
call `colab.zip_results()` (last cell) regularly to download your results.

In [ ]:
from fuguard_repro.config import IMAGE_DATASETS, METHODS, PROTOCOLS
from fuguard_repro.runner import run_grid, standard_jobs

DATASETS = IMAGE_DATASETS
SELECTED_METHODS = METHODS
SEEDS = PROTOCOLS["table2"]["seeds"]                      # (0, 1)
RECOVERY_ROUNDS = PROTOCOLS["table2"]["recovery_rounds"]  # 5

run_grid("table2", DATASETS, standard_jobs(SELECTED_METHODS), SEEDS, RECOVERY_ROUNDS)

In [ ]:
from fuguard_repro.aggregate import table2_numeric, table2_df
display(table2_df(table2_numeric()))

**What to check:** the FedAvg row should land near the paper's (SVHN 83.9 / CIFAR-10 64.9 / CIFAR-100 45.1 / EuroSAT 73.8 % ACC).
If it does not, fix the data/training side before looking at unlearning. Full comparison table → `03_analysis/03a_tables.ipynb`.

# 02b · Protocol B - 50 training rounds + **50 recovery rounds**
Produces **Table III** (rounds needed to reach 95 % / 100 % of the pre-unlearning accuracy). The paper reports values up to 50
(Retrain, NoT) which cannot come from the 5-round protocol, hence this separate run. Seed 0 only ("fixed random seed" in the paper).

Resumable: finished (dataset, method, seed) results in `results/raw/` are skipped, so you can stop and re-run.
The 49 shared training rounds are trained **once per (dataset, seed)** and cached in `artifacts/checkpoints/`;
every method then starts from the same pre-unlearning model (identical to what the repo's scripts recompute each time).

Entries marked `*` in the table mean "not reached within 50 rounds" (value shown = cap).

In [ ]:
from fuguard_repro.config import IMAGE_DATASETS, METHODS, PROTOCOLS
from fuguard_repro.runner import run_grid, standard_jobs

DATASETS = IMAGE_DATASETS
SEEDS = PROTOCOLS["table3"]["seeds"]                      # (0,)
RECOVERY_ROUNDS = PROTOCOLS["table3"]["recovery_rounds"]  # 50

run_grid("table3", DATASETS, standard_jobs(METHODS), SEEDS, RECOVERY_ROUNDS)

In [ ]:
from fuguard_repro.aggregate import table3_numeric, table3_df
display(table3_df(table3_numeric()))

# 02c · Ablation (Table IV, SVHN)
Rows: λ_OT ∈ {0, 0.1} × {only unlearn, with recovery}. This notebook runs **λ_OT = 0** only; the λ_OT = 0.1 row is read from the
Protocol-A FuGuard run (run 02a for SVHN/FuGuard first). "Only unlearn" = metrics measured right after the unlearning step (before any recovery round).

In [ ]:
from fuguard_repro.config import ot_variant
from fuguard_repro.runner import run_grid
run_grid("ablation", ["SVHN"], [(ot_variant(0.0), "fuguard", {"ot_lambda": 0.0})], seeds=(0, 1), recovery_rounds=5)

In [ ]:
from fuguard_repro.aggregate import table4_df
table4_df()

# 02d · λ_OT sensitivity (Fig. 10)
FuGuard with λ_OT ∈ {0.08, 0.09, 0.11, 0.12} on the four datasets (λ_OT = 0.10 is read from the Protocol-A run). Seeds {0, 1}, 5 recovery rounds.
Note: the CIFAR-10 panel of the paper's Fig. 10 shows ACC≈0.69 / ASR≈0.14 at λ=0.1, which differs from Table II (63.8 / 7.67 %) - the setting behind Fig. 10 is not stated, so expect a level offset.

In [ ]:
from fuguard_repro.config import IMAGE_DATASETS, ot_variant
from fuguard_repro.aggregate import SENS_LAMBDAS
from fuguard_repro.runner import run_grid

jobs = [(ot_variant(l), "fuguard", {"ot_lambda": l}) for l in SENS_LAMBDAS if abs(l - 0.1) > 1e-12]
run_grid("sensitivity", IMAGE_DATASETS, jobs, seeds=(0, 1), recovery_rounds=5)

In [ ]:
from fuguard_repro.aggregate import sensitivity_numeric
{ds: sensitivity_numeric(ds) for ds in IMAGE_DATASETS}

# 02e · Scalability: SVHN with 100 clients (Fig. 9)
Needs the 100-client data from `01a`. Assumptions (not in the paper): α = 0.1, attacker = `client_3`.
Results are stored under tag `SVHN-100`.

In [ ]:
from fuguard_repro.config import METHODS
from fuguard_repro.runner import run_grid, standard_jobs
run_grid("scalability", ["SVHN"], standard_jobs(METHODS), seeds=(0, 1), recovery_rounds=5,
         base_overrides={"num_clients": 100, "tag": "SVHN-100"})

# 02f · AWE real dataset (Fig. 7, Fig. 8)
6 clients, 2-layer bi-LSTM, 100 training epochs (rounds) + 50 recovery epochs, local lr 5e-5 (Adam), batch 32, 2 local epochs.
Compared: Retrain, FedSGA, FuGuard at unlearning lr ∈ {0.1, 0.05, 0.01}.

Fixes vs the repo: config keys; the repo ran with **no seed** (`CONFIG.get("seed")` → None); FedSGA's SGA loop in `main_awe/FedSGA.py` iterated
`train_dataset` (the previously trained client's data) instead of the target client's `target_dataset`; FuGuard encoded only 100 samples but kept all labels.
FuGuard trains its LSTM-VAE for 1000 epochs on the proxy subset *inside every unlearning call* (as in the repo), which is the slow part.
**Untested on the real data.**

In [ ]:
from fuguard_repro.config import lr_variant
from fuguard_repro.runner import run_grid

LRS = (0.1, 0.05, 0.01)
jobs = [("retrain", "retrain", {})]
for lr in LRS:
    jobs += [(lr_variant("fedsga", lr), "fedsga", {"forgetting_lr": lr}),
             (lr_variant("fuguard", lr), "fuguard", {"forgetting_lr": lr})]
run_grid("awe", ["AWE"], jobs, seeds=(0,), recovery_rounds=50, kind="awe")

In [ ]:
from IPython.display import Image, display
from fuguard_repro import plotting
for p in (plotting.plot_awe_training(), plotting.plot_awe_recovery(LRS)):
    if p: display(Image(str(p)))

# 03a · Tables: Table II, III, IV (ours vs paper)
Reads `results/raw/` and writes CSV + Markdown into `results/tables/`. "ours vs paper" tables give the difference in percentage points.
Std is the population std over seeds (`aggregate.STD_DDOF = 0`; the paper does not say which one it used).

In [ ]:
from fuguard_repro.reports import build_tables
T = build_tables()
for name, df in T.items():
    print(f"\n=== {name} ===")
    display(df)

# 03b · Figures (Fig. 6, 9, 10 + AWE Fig. 7/8)
Styles replicate the images in the repo's `results/` folder (same file names: `cifar10-acc.png`, `svhn-100-mia.png`, `eurosat-ot.png`, …).
Shaded bands = mean ± std over seeds. Outputs → `results/figures/`.

In [ ]:
from IPython.display import Image, display
from fuguard_repro.reports import build_figures
paths = build_figures()
for p in paths: print(p.name)

In [ ]:
from fuguard_repro.paths import FIGURES_DIR
display(Image(str(FIGURES_DIR / "fig6_recovery_grid.png")))

### Download everything (Colab)
Zips `results/` (+ cached checkpoints) and downloads it to your computer.

In [ ]:
from fuguard_repro import colab
colab.zip_results()

# Save your work

In [ ]:
colab.zip_results()                 # downloads results + checkpoints as a zip
# colab.backup_to_drive()           # OPTIONAL: copy them to your Google Drive instead